# Phase 1: Auditable Incident Deduplication

This notebook is the analyst-facing view of the Phase 1 v2 pipeline. It preserves every source report, consolidates only full-row exact and normalized-exact duplicates, and treats temporal proximity as review-only evidence.

`Date/Time` is interpreted as UTC under the approved source contract. Crossing-local timestamps are derived from the current Form 71 coordinate and historical IANA daylight-saving rules; they do not prove physical-event timing or historical crossing location.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    """Find the repository by its project layout, not its hidden Git metadata."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'analysis').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError(
        f'Could not find the project root from {start}. Open the notebook from the '
        'Blocked-Crossing-Prediction repository or set repo_root explicitly.'
    )

repo_root = find_repo_root(Path.cwd())
analysis_dir = repo_root / 'analysis'
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

from incident_deduplication import run_phase_1

authoritative_path = repo_root / 'data' / 'blocked_crossings_2020through2025.xlsx'
reconciliation_path = repo_root / 'data' / 'blocked_crossings_2025.xlsx'
inventory_path = repo_root / 'data' / 'Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv'
config_path = analysis_dir / 'incident_deduplication_config.json'
output_dir = repo_root / 'analysis_outputs' / 'deduplication' / 'v2'

In [2]:
result = run_phase_1(
    authoritative_path, reconciliation_path, inventory_path, config_path, output_dir
)
print(json.dumps(result.summary, indent=2))
assert result.validations['source_rows_map_once']
assert all(result.validations['raw_fingerprints_unchanged_after_processing'].values())

[1/9] Initializing output directory & verifying input file hashes...


ValueError: Authoritative workbook fingerprint differs from the configured approved input: expected e1a4a457c38fdb7e3f749d5fbc5f35fc436a9af37457acd34d72a710c9fa89f6, found a4b06d950f20acf8c4eb03930c43b27d399312277b92bd6412d63038de3cfcc1. Review and update the configuration before processing.

## Inventory, normalization, and provenance

Unknown duration values remain unmapped; invalid crossing IDs and timestamps remain in the source table and receive documented exceptions rather than canonical incidents.

In [ ]:
inventory_profile = json.loads((output_dir / 'inventory_profile.json').read_text(encoding='utf-8'))
pd.DataFrame([inventory_profile['duration_normalization'], inventory_profile['crossing_id_status']], index=['duration status', 'crossing ID status']).T.fillna(0)

## UTC and crossing-local time

UTC remains the canonical comparison timestamp. Rows without a coordinate-derived IANA zone remain UTC-only and are flagged; no state-level fallback is used.

In [ ]:
timezone_coverage = pd.read_csv(output_dir / 'timezone_assignment_diagnostics.csv')
local_time_diagnostics = pd.read_csv(output_dir / 'local_time_diagnostics.csv')
display(timezone_coverage)
local_time_diagnostics.sort_values(['iana_time_zone', 'reported_local_hour']).head(30)

In [ ]:
source_reports = pd.read_parquet(output_dir / 'source_reports_with_ids.parquet')
source_reports.loc[source_reports['timezone_assignment_status'].eq('assigned'), [
    'source_row_id', 'norm_crossing_id', 'reported_at_utc', 'reported_at_local',
    'iana_time_zone', 'utc_offset_minutes', 'State', 'City'
]].head(20)

## Conservative consolidation and review-only temporal candidates

Temporal proximity does not alter canonical assignments. The review sample is deterministic and must be labeled before Phase 1 can be marked complete.

In [ ]:
deduplication_summary = json.loads((output_dir / 'deduplication_summary.json').read_text(encoding='utf-8'))
display(pd.Series(deduplication_summary))
review_sample = pd.read_csv(output_dir / 'candidate_review_sample.csv')
review_sample.head(20)

## Reconciliation, diagnostics, and gate

The 2025 comparison is a normalized full-row multiset comparison, including duplicate multiplicity. A no-report interval must not be called unblocked.

In [ ]:
reconciliation_summary = json.loads((output_dir / 'reconciliation_summary.json').read_text(encoding='utf-8'))
gate_report = json.loads((output_dir / 'phase_1_gate_report.json').read_text(encoding='utf-8'))
display(pd.Series(reconciliation_summary))
display(pd.read_csv(output_dir / 'timestamp_granularity_by_year.csv'))
display(pd.read_csv(output_dir / 'diagnostics_by_year.csv'))
gate_report